# FIM Line Model — CodeT5-small, fill-in-the-middle

CodeT5 was pretrained on span denoising: `prefix <extra_id_0> suffix` → `<extra_id_0> middle <extra_id_1>`.
That already *is* fill-in-the-middle, so the model is fine-tuned in its native format instead of a new one
(`modules/fim.py`, `modules/datasets/fim_dataset.py`).

The hole never crosses a line break: the model finishes or fills **one line** while seeing the code above **and below** the cursor.
Hole types: `eol` (cursor → end of line), `inner` (the rest of the line, e.g. `):`, is already typed), `line` (a whole missing line).

In [ ]:
%tb
import os, math, random, glob
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")

from modules.Plotting import MetricLog, plot_metrics
from modules.hand_testing import hand_test_fim_repl
from modules.best_model_saver import BestModelSaver
from modules.early_stopping import EarlyStopping
from modules.datasets.loading import load_files
from modules.datasets.line_t5_dataset import collate_t5
from modules.datasets.fim_dataset import T5FIMDataset
from modules.evaluation_fim import evaluate_fim_model

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
# WORKDIR = r'/home/ubuntu/code_autocomplete/'
print(f"WORKDIR: {WORKDIR}")

LINE_MODEL_NAME = "line_model_FIM_T5_small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

## Training loop

In [ ]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def _t5_loss(model, src, mask, lbl, label_smoothing):
    # T5 does the label right-shift internally; we only override the loss
    logits = model(input_ids=src, attention_mask=mask, labels=lbl).logits
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), lbl.reshape(-1),
                           ignore_index=-100, label_smoothing=label_smoothing)
    return loss, logits


def train_line_model(
    model:           T5ForConditionalGeneration,
    train_dl:        DataLoader,
    val_dl:          DataLoader,
    epochs:          int,
    lr:              float,
    device:          torch.device,
    saver:           BestModelSaver,
    log:             MetricLog,
    plot_dir:        str,
    label_smoothing: float = 0.1,
    warmup_frac:     float = 0.05,
    patience:        int = 3,
    use_amp:         bool = True,
):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")

    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    # T5 is more stable in bf16 — use it if the GPU supports it, else fp16 + GradScaler
    amp_enabled = use_amp and device.type == "cuda"
    amp_dtype = torch.bfloat16 if (amp_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    scaler = GradScaler("cuda", enabled=(amp_enabled and amp_dtype == torch.float16))
    tqdm.write(f"[AMP] enabled={amp_enabled}, dtype={amp_dtype}")

    stopper = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        # TRAIN
        model.train()
        t_loss = t_steps = 0
        gn = 0.0
        batch_bar = tqdm(train_dl, desc=f"[Line] Epoch {ep}/{epochs} train", leave=False, unit="batch")
        for src, mask, lbl in batch_bar:
            src, mask, lbl = (t.to(device, non_blocking=True) for t in (src, mask, lbl))
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                loss, _ = _t5_loss(model, src, mask, lbl, label_smoothing)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                gn = _clip_norm(model)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                gn = _clip_norm(model)
                opt.step()
            sched.step()
            t_loss += loss.item()
            t_steps += 1
            batch_bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
        tl = t_loss / t_steps

        # VAL — token accuracy is measured here, on unseen files
        model.eval()
        v_loss = v_acc = v_steps = 0
        with torch.no_grad():
            for src, mask, lbl in tqdm(val_dl, desc=f"[Line] Epoch {ep}/{epochs} val  ", leave=False, unit="batch"):
                src, mask, lbl = (t.to(device, non_blocking=True) for t in (src, mask, lbl))
                with autocast("cuda", dtype=amp_dtype, enabled=amp_enabled):
                    loss, logits = _t5_loss(model, src, mask, lbl, label_smoothing)
                valid = (lbl != -100)
                v_acc += (logits.argmax(-1)[valid] == lbl[valid]).float().mean().item()
                v_loss += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        va = v_acc / v_steps if v_steps else 0.0

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"], token_acc=va, grad_norm=gn)
        tqdm.write(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
                   f"  ppl={math.exp(min(vl, 20)):.1f}  val_acc={va:.3f}"
                   f"  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [ ]:
class Arguments():
    def __init__(self,
                 data_dir=f"{WORKDIR}/Clean_Dataset",   # train/ val/ test/ written by DataPreparation
                 ckpt_dir=f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                 plot_dir=f"{WORKDIR}/plots/{LINE_MODEL_NAME}",
                 results_dir=f"{WORKDIR}/results/{LINE_MODEL_NAME}",
                 hf_model="Salesforce/codet5-small",
                 epochs=5, batch=32, lr=3e-4, max_files=0,
                 max_prefix=320, max_suffix=160, max_middle=64,   # tokens; 320 + 160 + 3 < 512
                 line_sample_rate=1.0,                           # < 1.0 to shorten an epoch
                 label_smoothing=0.1, warmup_frac=0.05, patience=3, use_amp=True,
                 seed=42, test=False, no_load=False):
        for k, v in locals().items():
            if k != "self": setattr(self, k, v)


def best_ckpt(ckpt_dir: str) -> Optional[str]:
    """Lowest val-loss checkpoint (file names sort by epoch, not by loss)."""
    paths = glob.glob(f"{ckpt_dir}/{LINE_MODEL_NAME}_*.pt")
    return min(paths, key=lambda p: float(p.rsplit("loss", 1)[1][:-3])) if paths else None


def main(args):
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)

    hf_tok = AutoTokenizer.from_pretrained(args.hf_model)
    model = T5ForConditionalGeneration.from_pretrained(args.hf_model).to(device)

    # test-only mode (no_load=True → the pretrained model, it already knows the format)
    if args.test:
        path = best_ckpt(args.ckpt_dir)
        if path and not args.no_load:
            model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
            print(f"[Loaded] line model from {path}")
        hand_test_fim_repl(model, hf_tok, device, kind="t5")
        return

    # separate folders from DataPreparation: test/ stays unseen until evaluation
    print("[Loading] Started loading")
    tr_txt = load_files(f"{args.data_dir}/train", args.max_files)
    va_txt = load_files(f"{args.data_dir}/val", args.max_files)
    if not tr_txt or not va_txt:
        print("[ERROR] no data — expected Clean_Dataset/train and Clean_Dataset/val")
        return
    print("[Loading] Ended loading")

    ds_kw = dict(max_prefix=args.max_prefix, max_suffix=args.max_suffix,
                 max_middle=args.max_middle, seed=args.seed)
    tr_ds = T5FIMDataset(tr_txt, hf_tok, line_sample_rate=args.line_sample_rate, **ds_kw)
    va_ds = T5FIMDataset(va_txt, hf_tok, deterministic=True, **ds_kw)
    collate = lambda b: collate_t5(b, hf_tok.pad_token_id)
    tr_dl = DataLoader(tr_ds, args.batch, shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
    va_dl = DataLoader(va_ds, args.batch, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Line  Model] {n_params/1e6:.1f}M parameters ({args.hf_model}, FIM)")

    saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME, from_hf=True)
    train_line_model(model, tr_dl, va_dl, args.epochs, args.lr, device, saver, MetricLog(), args.plot_dir,
                     args.label_smoothing, args.warmup_frac, args.patience, args.use_amp)


main(Arguments())
# main(Arguments(test=True))
# main(Arguments(test=True, no_load=True))

## Evaluation

On `Clean_Dataset/test` (never seen in training). Three runs:
the pretrained model (`NO-LOAD`, it already understands `<extra_id_0>`), the fine-tuned one,
and the fine-tuned one **without the suffix** — the ablation that shows what the code below the cursor gives.

In [ ]:
def evaluate(args, n_samples=500):
    hf_tok = AutoTokenizer.from_pretrained(args.hf_model)
    model = T5ForConditionalGeneration.from_pretrained(args.hf_model).to(device)
    test_txt = load_files(f"{args.data_dir}/test", 200)
    kw = dict(kind="t5", n_samples=n_samples, out_dir=args.results_dir, title="FIM Line - CodeT5-small")

    results = {"NO-LOAD": evaluate_fim_model(model, hf_tok, test_txt, device, tag="NO-LOAD ", **kw)}
    path = best_ckpt(args.ckpt_dir)
    if path:
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
        print(f"[Loaded] {path}")
        results["fine-tuned"] = evaluate_fim_model(model, hf_tok, test_txt, device, **kw)
        results["fine-tuned, no suffix"] = evaluate_fim_model(model, hf_tok, test_txt, device,
                                                             use_suffix=False, tag="NO-SUFFIX ", **kw)
    for name, r in results.items():
        print(f"  {name:22s} exact={r['exact_match']:.3f}  edit={r['edit_similarity']:.3f}  "
              f"syntax={r['syntax_valid']:.3f}  first_tok={r['first_token_match']:.3f}  "
              f"p50={r['latency_p50']:.0f} ms")
    return results


fim_results = evaluate(Arguments())